# Experiment 1.3b-ii-B: Thresholded Near-Zero Hessian Counts vs Linear Gauge-Dimension Prediction

## Counterpart and scope
- **Counterpart script:** `run_experiment.py` in the same directory.
- **Design choice in this first completion pass:** the notebook imports the script and uses its reusable runner rather than re-implementing the experiment core.
- **Primary observable actually measured here:** counts of Hessian eigenvalues satisfying
  \(|\lambda| < 	au \cdot \max |\lambda|\) for relative thresholds \(	au \in \{10^{-2}, 10^{-3}, 10^{-4}\}\).
- **Primary comparison actually made here:** in selected toy **deep linear exact-minimum** cases, compare those thresholded near-zero counts against the linear gauge-dimension prediction \(d^2(L-1)\).

## Truthful framing
This notebook does **not** directly measure a general “network symmetry count.” It does **not** construct gauge tangent bases, prove null-eigenspace equivalence, or analyze Muon updates. The ReLU section is therefore treated as **exploratory nonlinear comparison**, not as a direct symmetry-dimension measurement.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string(index=False))
        else:
            print(obj)


REPO_NAME = "Muon_as_RG_Gauge_Fixing"
RELATIVE_EXPERIMENT_DIR = Path(
    "experiments/Tier_1_Core_Mechanism_Tests/"
    "Exp_1.3_Singular_Value_Spectrum_Evolution/"
    "1.3b_Spectrum_Gradient_vs_Weight_Spectrum_Correlation/"
    "1.3b-ii_Hessian_Alignment_Does_Muon_Update_Avoid_Flat_Directions/"
    "1.3b-ii-B_Hessian_Null_Space_Dimension_vs_Symmetry_Count"
)


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == REPO_NAME and (candidate / RELATIVE_EXPERIMENT_DIR).exists():
            return candidate
        if (candidate / RELATIVE_EXPERIMENT_DIR).exists():
            return candidate
    fallback = Path("/home/secemp9/Muon_as_RG_Gauge_Fixing")
    if (fallback / RELATIVE_EXPERIMENT_DIR).exists():
        return fallback
    raise FileNotFoundError(
        "Could not locate the repository root containing the counterpart script."
    )


PROJECT_ROOT = resolve_project_root()
EXPERIMENT_DIR = PROJECT_ROOT / RELATIVE_EXPERIMENT_DIR
SCRIPT_PATH = EXPERIMENT_DIR / "run_experiment.py"

spec = importlib.util.spec_from_file_location("exp_1_3b_ii_B", SCRIPT_PATH)
experiment = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = experiment
spec.loader.exec_module(experiment)

print(f"Project root : {PROJECT_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Script path   : {SCRIPT_PATH}")
print(f"Imported module exposes run_full_experiment: {hasattr(experiment, 'run_full_experiment')}")

## Reproducibility, configuration, and runtime notes
The default script plan is preserved in this notebook:
- linear exact minima: `(4,2)`, `(4,3)`, `(4,4)`, `(6,3)`
- linear random-init controls: `(4,2)`, `(4,3)`
- exploratory ReLU runs: `(4,2)`, `(4,3)`
- synthetic sample count: `n_samples = 20`
- dataset seed per config: `100 + 10*d + L`
- model/gauge seed: `42`
- finite-difference steps: gradient `1e-6`, Hessian `1e-5`
- null-count thresholds: `1%`, `0.1%`, `0.01%` of `max |eigenvalue|`

Runtime caveat: this is a pure NumPy finite-difference Hessian study. A full run is much slower than a smoke run because the Hessian cost grows roughly like `2P^2 + 1` loss evaluations for `P = d^2 L`, and the exploratory ReLU training uses repeated finite-difference gradients.

In [ ]:
RUN_MODE = globals().get("RUN_MODE", "full")  # "full" or "smoke"
RUN_RELU = globals().get("RUN_RELU", True)
PRINT_PROGRESS = globals().get("PRINT_PROGRESS", False)
RELU_VERBOSE = globals().get("RELU_VERBOSE", False)
SELECTED_SPECTRA_LABELS = globals().get(
    "SELECTED_SPECTRA_LABELS",
    [
        "LINEAR@exact-min d=4,L=2",
        "LINEAR@exact-min d=4,L=3",
        "LINEAR@rand d=4,L=2",
    ],
)

plan = (
    experiment.get_smoke_experiment_plan(relu_verbose=RELU_VERBOSE)
    if RUN_MODE == "smoke"
    else experiment.get_default_experiment_plan(relu_verbose=RELU_VERBOSE)
)
if not RUN_RELU:
    plan["relu_configs"] = []

print(f"Run mode      : {RUN_MODE}")
print(f"Include ReLU  : {RUN_RELU}")
print(f"Print progress: {PRINT_PROGRESS}")
print(f"Plan name     : {plan['plan_name']}")
print(f"Thresholds    : {[experiment.threshold_to_display(t) for t in plan['null_thresholds']]}")
print(f"Linear minima : {plan['linear_min_configs']}")
print(f"Linear control: {plan['linear_control_configs']}")
print(f"ReLU configs  : {plan['relu_configs']}")

runtime_rows = []
for group_name, configs in [
    ("linear_exact_minimum", plan["linear_min_configs"]),
    ("linear_random_control", plan["linear_control_configs"]),
    ("relu_trained_exploratory", plan["relu_configs"]),
]:
    for d, L in configs:
        p = d * d * L
        row = {
            "group": group_name,
            "config": f"d={d},L={L}",
            "params_P": p,
            "hessian_loss_evals_approx": 2 * p * p + 1,
        }
        if group_name == "relu_trained_exploratory":
            row["gradient_loss_evals_per_step"] = 2 * p
            row["max_training_loss_evals"] = 2 * p * plan["relu_train_kwargs"]["n_steps"]
        runtime_rows.append(row)

runtime_df = pd.DataFrame(runtime_rows)
print()
print("Approximate finite-difference cost summary:")
print(runtime_df.to_string(index=False))

## Execute the script-backed experiment
This is the notebook-native execution cell. It deliberately avoids the old `if __name__ == '__main__': main()` pattern. By default, the cell uses the full script plan; for quick checks, set `RUN_MODE = "smoke"` in the configuration cell above.

In [ ]:
bundle = experiment.run_full_experiment(plan=plan, print_progress=PRINT_PROGRESS)

print(bundle["title"])
print(bundle["scope"])
print(f"Completed {len(bundle['results'])} configurations.")
print("Groups:", {k: len(v) for k, v in bundle["groups"].items()})

## Compact table of all configurations and diagnostics
The table below is the main notebook summary. It reports the predicted linear gauge-dimension value, observed thresholded near-zero counts, gradient norm at the analyzed point, negative-eigenvalue count, Hessian asymmetry, and elapsed Hessian time.

In [ ]:
thresholds = bundle["plan"]["null_thresholds"]
summary_df = pd.DataFrame(bundle["summary_rows"]).sort_values(
    ["group", "d", "L", "label"]
).reset_index(drop=True)

summary_cols = [
    "group",
    "label",
    "d",
    "L",
    "total_params",
    "predicted_gauge_dim",
    "loss_value",
    "grad_norm",
    "reconstruction_error",
    "n_negative",
    "hessian_asymmetry",
    "elapsed_s",
]
for threshold in thresholds:
    key = experiment.threshold_to_key(threshold)
    summary_cols.extend([
        f"null_count_{key}",
        f"null_ratio_{key}",
    ])

summary_view = summary_df[summary_cols].copy()
rename_map = {
    "group": "group",
    "label": "label",
    "d": "d",
    "L": "L",
    "total_params": "params",
    "predicted_gauge_dim": "pred_linear_null_dim",
    "loss_value": "loss",
    "grad_norm": "grad_norm",
    "reconstruction_error": "recon_err",
    "n_negative": "n_negative",
    "hessian_asymmetry": "hessian_asym",
    "elapsed_s": "elapsed_s",
}
for threshold in thresholds:
    key = experiment.threshold_to_key(threshold)
    disp = experiment.threshold_to_display(threshold)
    rename_map[f"null_count_{key}"] = f"null_count@{disp}"
    rename_map[f"null_ratio_{key}"] = f"ratio@{disp}"
summary_view = summary_view.rename(columns=rename_map)

print(summary_view.to_string(index=False))
display(summary_view)

warning_rows = []
for result in bundle["results"]:
    for warning in result["warnings"]:
        warning_rows.append({
            "label": result["label"],
            "group": result["group"],
            "warning": warning,
        })

if warning_rows:
    warnings_df = pd.DataFrame(warning_rows)
    print()
    print("Warnings / interpretation caveats raised by the script:")
    print(warnings_df.to_string(index=False))
    display(warnings_df)
else:
    print()
    print("No automatic warnings were raised by the current run.")

## Selected Hessian spectra
The next figure visualizes sorted absolute Hessian eigenvalues for selected configurations. The red dashed line marks the **predicted linear gauge boundary** at index `d^2(L-1)`. For deep linear exact-minimum cases, a useful qualitative sign is whether many of the smallest eigenvalues cluster near zero before that boundary.

In [ ]:
selected_results = []
for label in SELECTED_SPECTRA_LABELS:
    match = next((r for r in bundle["results"] if r["label"] == label), None)
    if match is not None:
        selected_results.append(match)

if not selected_results:
    raise ValueError("No selected spectra labels were found in the current result bundle.")

fig, axes = plt.subplots(
    1,
    len(selected_results),
    figsize=(6 * len(selected_results), 4.5),
    constrained_layout=True,
)
if len(selected_results) == 1:
    axes = [axes]

for ax, result in zip(axes, selected_results):
    abs_eigs = np.maximum(result["eigenvalues_abs_sorted"], 1e-16)
    idx = np.arange(len(abs_eigs))
    ax.semilogy(idx, abs_eigs, marker="o", linewidth=1.0, markersize=3)
    boundary = result["predicted_gauge_dim"] - 0.5
    ax.axvline(boundary, color="crimson", linestyle="--", linewidth=1.5)
    ax.set_title(result["label"])
    ax.set_xlabel("sorted index")
    ax.set_ylabel(r"$|\lambda|$")
    ax.grid(True, which="both", alpha=0.3)

plt.show()

## Observed vs predicted near-zero counts for linear minima and controls
This section keeps the central comparison honest. The prediction `d^2(L-1)` is a **linear gauge-dimension benchmark**. The bars are **observed thresholded near-zero counts**, not exact nullities.

In [ ]:
linear_results = [
    r for r in bundle["results"]
    if r["group"] in {"linear_exact_minimum", "linear_random_control"}
]

comparison_rows = []
for result in linear_results:
    row = {
        "group": result["group"],
        "label": result["label"],
        "predicted_gauge_dim": result["predicted_gauge_dim"],
        "grad_norm": result["grad_norm"],
        "n_negative": result["n_negative"],
    }
    for threshold in thresholds:
        row[f"obs_{experiment.threshold_to_key(threshold)}"] = result["null_counts"][threshold]
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
display(comparison_df)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False, constrained_layout=True)
group_order = ["linear_exact_minimum", "linear_random_control"]
bar_width = 0.22
x_offsets = np.arange(len(thresholds)) - (len(thresholds) - 1) / 2

for ax, group_name in zip(axes, group_order):
    group_df = comparison_df[comparison_df["group"] == group_name].reset_index(drop=True)
    x = np.arange(len(group_df))
    for offset, threshold in zip(x_offsets, thresholds):
        obs_col = f"obs_{experiment.threshold_to_key(threshold)}"
        ax.bar(
            x + offset * bar_width,
            group_df[obs_col],
            width=bar_width,
            label=f"observed @ {experiment.threshold_to_display(threshold)}",
        )
    ax.plot(
        x,
        group_df["predicted_gauge_dim"],
        color="black",
        marker="x",
        linestyle="none",
        markersize=8,
        label="predicted linear gauge dim",
    )
    ax.set_title(experiment.GROUP_TITLES[group_name])
    ax.set_ylabel("count")
    ax.set_xticks(x)
    ax.set_xticklabels(group_df["label"], rotation=20, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best")

plt.show()

## Exploratory nonlinear / ReLU comparison
This section is intentionally cautious. The script trains small ReLU networks toward low loss using finite-difference gradient descent and then analyzes the Hessian at the final endpoint. The notebook therefore reports these runs as **exploratory** and checks whether the training endpoint is plausibly near criticality before drawing any qualitative comparison to the linear exact-minimum cases.

In [ ]:
relu_results = bundle["groups"]["relu_trained_exploratory"]
linear_lookup = {
    (r["d"], r["L"]): r for r in bundle["groups"]["linear_exact_minimum"]
}

if relu_results:
    relu_rows = []
    for result in relu_results:
        matched_linear = linear_lookup.get((result["d"], result["L"]))
        row = {
            "label": result["label"],
            "d": result["d"],
            "L": result["L"],
            "loss": result["loss_value"],
            "grad_norm": result["grad_norm"],
            "n_negative": result["n_negative"],
            "stop_reason": None if result["training_info"] is None else result["training_info"]["stop_reason"],
            "steps_completed": None if result["training_info"] is None else result["training_info"]["steps_completed"],
        }
        for threshold in thresholds:
            row[f"relu_null@{experiment.threshold_to_display(threshold)}"] = result["null_counts"][threshold]
            row[f"linear_match@{experiment.threshold_to_display(threshold)}"] = (
                None if matched_linear is None else matched_linear["null_counts"][threshold]
            )
        relu_rows.append(row)

    relu_df = pd.DataFrame(relu_rows)
    print(relu_df.to_string(index=False))
    display(relu_df)

    near_critical_mask = (relu_df["grad_norm"] < 1e-5) & (relu_df["n_negative"] == 0)
    print()
    print(
        "Near-critical heuristic (grad_norm < 1e-5 and n_negative == 0):",
        near_critical_mask.tolist(),
    )
    if not bool(near_critical_mask.all()):
        print(
            "Interpretation note: at least one exploratory ReLU endpoint fails the notebook's simple "
            "near-critical heuristic, so any null-count comparison should be treated as suggestive or inconclusive."
        )
else:
    print("No ReLU runs are present in the current plan (for example, a smoke or no-ReLU run).")

## Final conclusion template
When interpreting a run of this notebook, separate the claims as follows:

### Established by this experiment when the tables/figures support it
- The code measures **thresholded near-zero Hessian counts** in selected toy configurations.
- In deep linear exact-minimum cases, those counts can be compared against the **linear gauge-dimension prediction** `d^2(L-1)`.
- Random linear initializations provide a control showing what the same metric looks like away from the constructed exact-minimum setting.

### Suggestive but not proved here
- If linear exact-minimum runs show a substantial block of very small eigenvalues near the predicted boundary, that is **consistent with** gauge-flat directions in this toy setting.
- If exploratory ReLU runs show smaller near-zero counts than matched linear exact-minimum runs, that is **suggestive** of reduced or altered flat-direction structure, but not a direct symmetry-dimension measurement.

### Untested by this notebook/script pair
- direct construction of gauge tangent vectors and overlap with the numerically small-eigenvalue subspace
- robustness sweeps over finite-difference step sizes and thresholds
- multi-seed uncertainty quantification
- any direct Muon-versus-SGD update alignment analysis

In [ ]:
primary_threshold = thresholds[0]
linear_exact = bundle["groups"]["linear_exact_minimum"]
linear_control = bundle["groups"]["linear_random_control"]
relu_runs = bundle["groups"]["relu_trained_exploratory"]

print("Established / suggestive / untested summary for the current run")
print("-" * 72)
print(f"Primary reporting threshold: {experiment.threshold_to_display(primary_threshold)}")

if linear_exact:
    consistent = [
        r for r in linear_exact
        if 0.8 <= r["null_ratios_to_prediction"][primary_threshold] <= 1.2
    ]
    print(
        f"Established (linear exact-minimum evidence base): {len(consistent)}/{len(linear_exact)} "
        "runs fall in an observed/predicted ratio band of 0.8-1.2 at the loosest threshold."
    )

if linear_control:
    control_counts = [r["null_counts"][primary_threshold] for r in linear_control]
    print(
        "Control observation (linear random-init): observed near-zero counts at the primary threshold =",
        control_counts,
    )

if relu_runs:
    relu_grad_norms = [r["grad_norm"] for r in relu_runs]
    print("Exploratory observation (ReLU grad norms):", relu_grad_norms)
    print("Treat any ReLU interpretation as exploratory unless the endpoint diagnostics are strong.")

print("Untested here: gauge tangent construction, step-size robustness sweeps, multi-seed statistics, Muon update analysis.")